In [0]:
ECOMM_BRONZE_FILES = "abfss://bronze@gopaldatalake.dfs.core.windows.net/ecomm-gopal/"

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType

# Step 1: Define explicit schema
ecomm_schema = StructType([
    StructField("InvoiceNo", StringType(), True),
    StructField("StockCode", StringType(), True),
    StructField("Description", StringType(), True),
    StructField("Quantity", IntegerType(), True),
    StructField("InvoiceDate", StringType(), True), # the csv does not have sql datetime format
    StructField("UnitPrice", DoubleType(), True),
    StructField("CustomerID", IntegerType(), True),
    StructField("Country", StringType(), True)
])

# LAZY , not yet started the stream
# Step 1: Read from Bronze Zone
bronze_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .schema(ecomm_schema)
    .load(ECOMM_BRONZE_FILES)  
)

# .show will not work on stream
bronze_df.printSchema()
bronze_df.show(2)

In [0]:
%sql 

DROP TABLE IF EXISTS orderdb.gopal_ecomm_sorted

In [0]:
(
bronze_df
.orderBy("UnitPrice")
.repartition(8)
.write
.format("delta")
.mode("overwrite")
.saveAsTable("orderdb.gopal_ecomm_sorted")
)

In [0]:
%sql
EXPLAIN FORMATTED
SELECT *
FROM orderdb.gopal_ecomm_sorted
WHERE UnitPrice > 200

-- check for PushedFilters: [IsNotNull(UnitPrice), GreaterThan(UnitPrice,200.0)] 
-- storage layer receive UnitPrice > 200
-- stats missing  missing = gopal_ecomm_sorted optimizer stats absent


-- == Optimizer Statistics (table names per statistics state) ==
--   missing = gopal_ecomm_sorted
--   partial = 
--   full    = 


In [0]:
%sql 

-- optimizer statistics for Spark’s cost-based optimizer (CBO) not related to data skipping
-- better join plans, cardinality estimates, filter selectivity estimates, some pruning decision
ANALYZE TABLE orderdb.gopal_ecomm_sorted
COMPUTE STATISTICS FOR ALL COLUMNS

-- for all columns

In [0]:
%sql
EXPLAIN FORMATTED
SELECT *
FROM orderdb.gopal_ecomm_sorted
WHERE UnitPrice > 200

-- check for stats



-- == Optimizer Statistics (table names per statistics state) ==
--   missing = 
--   partial = 
--   full    = gopal_ecomm_sorted

In [0]:
%sql
-- stats helpful in making decision for join order and broadcast 

EXPLAIN FORMATTED
SELECT *
FROM orderdb.gopal_ecomm_sorted e
JOIN (
  SELECT DISTINCT Country
  FROM orderdb.gopal_ecomm_sorted
) c
ON e.Country = c.Country

In [0]:
%sql

DESCRIBE DETAIL orderdb.gopal_ecomm_sorted

-- check numFiles, sizeInBytes

In [0]:
%sql
SELECT *
FROM orderdb.gopal_ecomm_sorted
WHERE UnitPrice > 200

In [0]:
%sql
EXPLAIN FORMATTED
SELECT *
FROM orderdb.gopal_ecomm_sorted
WHERE UnitPrice > 200

-- PushedFilters: [IsNotNull(UnitPrice), GreaterThan(UnitPrice,200.0)] as we know already